Basic workflow for making a "surrogate" model that learns the unpolarized BSA as function of $x_{\text{B}}$, $t$, $Q^{2}$, and $\phi$.

In [ ]:
import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import regularizers
from sklearn.model_selection import train_test_split

In [ ]:
plt.rcParams.update({
    "text.usetex": True, "font.family": "serif",
})
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['xtick.major.size'] = 8.5
plt.rcParams['xtick.major.width'] = 0.5
plt.rcParams['xtick.minor.size'] = 2.5
plt.rcParams['xtick.minor.width'] = 0.5
plt.rcParams['xtick.minor.visible'] = True
plt.rcParams['xtick.top'] = True
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['ytick.major.size'] = 8.5
plt.rcParams['ytick.major.width'] = 0.5
plt.rcParams['ytick.minor.size'] = 2.5
plt.rcParams['ytick.minor.width'] = 0.5
plt.rcParams['ytick.minor.visible'] = True
plt.rcParams['ytick.right'] = True
plt.rcParams['savefig.dpi'] = 300

In [ ]:
test_dataframe = pd.read_csv('../data/small_dset.csv')

In [ ]:
x_data = test_dataframe[["t", "x_b", "q_squared", "phi"]]
y_data = test_dataframe[["unp_target_bsa"]]

In [ ]:
x_data.head()

In [ ]:
y_data.head()

In [ ]:
x_remaining, x_testing, y_remaining, y_testing = train_test_split(
    x_data, y_data,
    test_size = 0.1, shuffle = True)

x_training, x_validation, y_training, y_validation = train_test_split(
    x_remaining, y_remaining,
    test_size = 0.1, shuffle = True)

In [ ]:
len(x_training)

In [ ]:
len(x_validation)

In [ ]:
len(x_testing)

In [ ]:
def cff_h_model():
    # initializer:
    initializer = tf.keras.initializers.GlorotNormal(seed = None)

    # regularizer:
    # I learned about regularizers here: https://medium.com/@theo.wolf/physics-informed-neural-networks-a-simple-tutorial-with-pytorch-f28a890b874a
    weight_regularizer = regularizers.l2(0.01) 
    
    # input layer:
    model_inputs = tf.keras.Input(shape = (4,), name = "input_values")

    # hidden layers:
    hidden = tf.keras.layers.Dense(
        32, kernel_initializer = initializer, kernel_regularizer = weight_regularizer, activation = "tanh")(model_inputs)
    hidden = tf.keras.layers.Dense(
        32, kernel_initializer = initializer, kernel_regularizer = weight_regularizer, activation = "tanh")(hidden)

    # output layer:
    model_output = tf.keras.layers.Dense(1, activation = "linear")(hidden)

    model = tf.keras.Model(
        inputs = model_inputs,
        outputs = model_output)

    model.compile(
        optimizer = tf.keras.optimizers.Adam(learning_rate = 0.01),
        loss = tf.keras.losses.MeanSquaredError(),
        )
    return model

In [ ]:
tf.keras.backend.clear_session()

dnn_model = cff_h_model()

dnn_model_history = dnn_model.fit(
    x_training, y_training,
    validation_data = (x_validation, y_validation),
    epochs = 1000, batch_size = len(x_training),
    verbose = 1)

In [ ]:
model_testing_loss = dnn_model.evaluate(x_testing, y_testing, verbose = 0)
print(f"[INFO]: Test Loss: {model_testing_loss}")

In [ ]:
figure, axis = plt.subplots(1, 1, figsize = (6, 6))

axis.plot(dnn_model_history.history['loss'], 
    label = "Training Loss", color = 'orange', alpha = 0.6)
axis.plot(dnn_model_history.history['val_loss'], 
    label = "Validation Loss", color = 'purple', alpha = 0.6)

axis.set_xlabel("Epoch", fontsize = 14.)
axis.set_ylabel("Loss Values", fontsize = 14.)
axis.set_title(rf"BSA Surrogate Model (Testing = ${model_testing_loss:.3f}$)",
    fontsize = 14.)
axis.legend(fontsize = 14.)
axis.grid(visible = False)

axis.text(
    0.00, -0.11,
    f"Figure rendered {datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}", 
    transform = axis.transAxes,
    verticalalignment = 'top',
    horizontalalignment = 'left',
    fontsize = 9.,
)

for extension in ['png', 'eps']:
    figure.savefig(
        fname = f"./plots/bsa_surrogate_test_v2.{extension}",
        facecolor = 'white',
        transparent = False)

plt.close(figure)

In [ ]:
predictions = dnn_model.predict(x_data).flatten()
actual = test_dataframe["unp_target_bsa"].values
phi_values = test_dataframe["phi"].values
residuals = actual - predictions

sum_squared_residuals = np.sum(residuals**2)
reduced_chi = sum_squared_residuals / len(phi_values)

residuals_figure, (ax1, ax2) = plt.subplots(2, 1, figsize = (10, 8), sharex = True, 
                               gridspec_kw = {'height_ratios': [3, 1]})

# Top Panel: Actual vs Predicted
ax1.scatter(phi_values, actual, color = 'black', label = 'Experimental Data', alpha = 0.7)
ax1.plot(phi_values, predictions, color = 'red', lw = 2, label = 'DNN Prediction')
ax1.set_xlabel(r"$\phi$ (radians)", fontsize = 16.)
ax1.set_ylabel(r"$\textrm{BSA}(\Lambda = 0)$", fontsize = 16.)
ax1.set_title(rf"BSA Surrogate Model (Testing = ${model_testing_loss:.3f}$)", fontsize = 14.)
ax1.legend()
ax1.grid(True, linestyle = ':', alpha = 0.6)

# Bottom Panel: Residuals
ax2.scatter(phi_values, residuals, color = 'blue', alpha = 0.6)
ax2.axhline(0, color = 'black', linestyle = '--')
ax2.set_ylabel(rf'Residuals ($\chi^2_\nu = {reduced_chi:.3f}$)', fontsize = 16.)
ax2.set_xlabel(r"$\phi$ (radians)", fontsize = 16.)
ax2.grid(True, linestyle = ':', alpha = 0.6)

for extension in ['png', 'eps']:
    residuals_figure.savefig(
        fname = f"./plots/bsa_surrogate_residuals_v2.{extension}",
        facecolor = 'white',
        transparent = False)

plt.close(residuals_figure)